# SmokeSignal AI: Robustness & Out-of-Distribution (OOD) Testing Report

## Executive Summary
This report documents the robustness of the SmokeSignal AI wildfire detection model against edge cases and its ability to detect Out-of-Distribution (OOD) inputs using the MobileNetV2-based scene classifier.

### Key Findings:
1. **OOD Detection Effectiveness**: The current keyword-based OOD detector successfully flags obvious non-satellite images (e.g., UI screenshots, text documents) but misses some abstract patterns (e.g., gradients).
2. **Model Sensitivity**: The wildfire detection model is highly sensitive to pure black images, erroneously classifying them as wildfires with 100% confidence. This indicates a potential bias toward dark pixels or lack of negative samples during training.
3. **Robustness to Noise**: Random noise is correctly classified as 'No Wildfire', though it isn't always flagged as OOD.

## Test Environment
- **Main Model**: `wildfire_detector_model.keras`
- **OOD Detector**: MobileNetV2 (ImageNet weights) with keyword filtering.
- **Test Cases**: Synthetic edge cases (Black, White, Noise, Gradient, Text) and internal report figures (Charts).

In [1]:
# Summary of results from tests/test_ood.py
results = [
    {"file": "gradient.jpg", "status": "ID ✅", "scene": "spotlight", "wildfire_conf": "78.51%"},
    {"file": "pure_black.jpg", "status": "OOD ⚠️", "scene": "digital clock", "wildfire_conf": "100.00%"},
    {"file": "pure_white.jpg", "status": "OOD ⚠️", "scene": "wall clock", "wildfire_conf": "0.39%"},
    {"file": "random_noise.jpg", "status": "ID ✅", "scene": "wool", "wildfire_conf": "5.72%"},
    {"file": "text_image.jpg", "status": "OOD ⚠️", "scene": "envelope", "wildfire_conf": "0.32%"},
    {"file": "class_distribution.png", "status": "ID ✅", "scene": "binder", "wildfire_conf": "0.00%"},
    {"file": "confusion_matrix.png", "status": "OOD ⚠️", "scene": "web site", "wildfire_conf": "0.00%"}
]

import pandas as pd
df = pd.DataFrame(results)
df

,file,status,scene,wildfire_conf
0,gradient.jpg,ID ✅,spotlight,78.51%
1,pure_black.jpg,OOD ⚠️,digital clock,100.00%
2,pure_white.jpg,OOD ⚠️,wall clock,0.39%
3,random_noise.jpg,ID ✅,wool,5.72%
4,text_image.jpg,OOD ⚠️,envelope,0.32%
5,class_distribution.png,ID ✅,binder,0.00%
6,confusion_matrix.png,OOD ⚠️,web site,0.00%


## Detailed Analysis

### 1. Robustness Issues
**Observation**: `pure_black.jpg` (simulating night or severe shadow) resulted in 100% wildfire confidence.
**Reasoning**: Many satellite wildfire datasets use IR channels or look for bright orange/red pixels. If the model was trained mostly on day imagery, pure black might be an unseen edge case that triggers high activation in the 'fire' neurons if they respond to extreme values or lack of background context.
**Recommendation**: Incorporate more 'Night' and 'Empty/Dark' satellite images into the training set as negative samples.

### 2. OOD Detection Gaps
**Observation**: `gradient.jpg` and `class_distribution.png` were considered 'In-Distribution'.
**Reasoning**: MobileNetV2 classified them as 'spotlight' and 'binder', which are not in the `OOD_KEYWORDS` list in `utils/ood.py`.
**Recommendation**: Expand the keyword list or use a more holistic OOD detection method (e.g., Mahalanobis distance or Energy-based OOD detection) that doesn't rely on specific labels.

## Conclusion
The system is robust against obvious UI/Screenshot noise but requires further work for specialized satellite edge cases like night imagery and abstract noise patterns.